In [12]:
import json
import pandas as pd
from pathlib import Path

In [13]:
album_df = pd.read_csv("../data/processed/album_tracks.csv")

raw_dir = Path("../data/raw/artists")
artists_files = list(raw_dir.glob("*.json"))

album_df.head()

,artist_id,artist_name,release_group_id,release_group_title,release_id,release_title,release_date,country,medium_format,track_position,track_title,recording_id,length_ms
0,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,1,Blew,2837e0d2-5ce9-44da-803e-797c835d672c,174133.0
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,2,Floyd the Barber,4cce1ee1-38e6-4ea0-8584-b6ebeb854d7d,137973.0
2,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,3,About a Girl,b2e623ea-378b-4479-bb75-fa202aacbfc9,168293.0
3,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,4,School,0b3292f6-8f0d-4160-92d3-49348029a619,162173.0
4,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,5,Love Buzz,1adfecb7-875f-4203-b3b1-8e2e643f94a2,215093.0


In [14]:
artist_albums = (
    album_df[["artist_id", "artist_name", "release_group_title"]]
    .drop_duplicates()
    .groupby(["artist_id", "artist_name"])["release_group_title"]
    .apply(list)
    .reset_index()
)

artist_albums

,artist_id,artist_name,release_group_title
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,[Ultramega OK]
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,[Bleach]
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,[Adrenaline]
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,[Ten]
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,[Gish]
5,ba853904-ae25-4ebb-89d6-c44cfbd71bd2,Blur,[Leisure]


In [15]:
tag_rows = []

for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)

    artist_id = artist.get("id")
    artist_name = artist.get("name")

    for tag in artist.get("tags", []):
        tag_rows.append({
            "artist_id": artist_id,
            "artist_name": artist_name,
            "tag": tag.get("name"),
            "tag_count": tag.get("count", 0)
        })

tag_df = pd.DataFrame(tag_rows)
tag_df.head()

,artist_id,artist_name,tag,tag_count
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,alternative metal,14
1,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,alternative rock,18
2,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,american,2
3,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,grunge,20
4,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,hard rock,9


In [16]:
top_tags = (
    tag_df.sort_values(["artist_id", "tag_count"], ascending=[True, False])
    .groupby(["artist_id", "artist_name"])
    .head(10)
    .groupby(["artist_id", "artist_name"])["tag"]
    .apply(list)
    .reset_index()
)

top_tags

,artist_id,artist_name,tag
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,"[grunge, alternative rock, alternative metal, ..."
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,"[grunge, alternative rock, rock, noise rock, a..."
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,"[alternative metal, nu metal, alternative rock..."
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,"[grunge, alternative rock, rock, hard rock, se..."
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,"[alternative rock, dream pop, neo-psychedelia,..."
5,ba853904-ae25-4ebb-89d6-c44cfbd71bd2,Blur,"[alternative rock, britpop, indie rock, britis..."


In [17]:
artist_texts = artist_albums.merge(
    top_tags,
    on=["artist_id", "artist_name"],
    how="left"
)

artist_texts["tag"] = artist_texts["tag"].apply(lambda x: x if isinstance(x, list) else [])
artist_texts

,artist_id,artist_name,release_group_title,tag
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,[Ultramega OK],"[grunge, alternative rock, alternative metal, ..."
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,[Bleach],"[grunge, alternative rock, rock, noise rock, a..."
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,[Adrenaline],"[alternative metal, nu metal, alternative rock..."
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,[Ten],"[grunge, alternative rock, rock, hard rock, se..."
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,[Gish],"[alternative rock, dream pop, neo-psychedelia,..."
5,ba853904-ae25-4ebb-89d6-c44cfbd71bd2,Blur,[Leisure],"[alternative rock, britpop, indie rock, britis..."


In [18]:
def build_embedding_text(row):
    albums = ", ".join(row["release_group_title"])
    tags = ", ".join(row["tag"])

    return f"Albums: {albums}. Tags: {tags}."

artist_texts["text_for_embedding"] = artist_texts.apply(build_embedding_text, axis=1)

artist_texts[["artist_name", "text_for_embedding"]]

,artist_name,text_for_embedding
0,Soundgarden,"Albums: Ultramega OK. Tags: grunge, alternativ..."
1,Nirvana,"Albums: Bleach. Tags: grunge, alternative rock..."
2,Deftones,"Albums: Adrenaline. Tags: alternative metal, n..."
3,Pearl Jam,"Albums: Ten. Tags: grunge, alternative rock, r..."
4,The Smashing Pumpkins,"Albums: Gish. Tags: alternative rock, dream po..."
5,Blur,"Albums: Leisure. Tags: alternative rock, britp..."


In [21]:
artist_texts_clean = artist_texts[["artist_id", "artist_name", "text_for_embedding"]]

out_path = Path("../data/processed/artist_texts.csv")
artist_texts_clean.to_csv(out_path, index=False)

out_path

WindowsPath('../data/processed/artist_texts.csv')

In [22]:
pd.read_csv("../data/processed/artist_texts.csv").head()

,artist_id,artist_name,text_for_embedding
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,"Albums: Ultramega OK. Tags: grunge, alternativ..."
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,"Albums: Bleach. Tags: grunge, alternative rock..."
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,"Albums: Adrenaline. Tags: alternative metal, n..."
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,"Albums: Ten. Tags: grunge, alternative rock, r..."
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,"Albums: Gish. Tags: alternative rock, dream po..."


## Phase 6A Result

Created clean artist-level text for vector retrieval.

The default embedding text includes:
- album / release group title
- MusicBrainz tags

The default embedding text excludes:
- member names
- membership relationships
- graph-only relationship facts

This avoids relational leakage and keeps the vector retrieval baseline fair.